In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

df = pd.read_csv('django_data.csv')

model = SentenceTransformer('all-MiniLM-L6-v2')

model.to('cuda')

emmbeddings = model.encode(df['Content'].tolist(), convert_to_tensor=True)

def ask_question(question, certainty_trashold=0.1):
    quesion_embedding = model.encode(question, convert_to_tensor=True)

    cos_scores = util.cos_sim(quesion_embedding, emmbeddings)[0]

    best_id = cos_scores.argmax().item()
    certainity = cos_scores[best_id].item()

    if certainity < certainty_trashold:
        return 'I dont know the answer to this question', '', ''

    theme = df.iloc[best_id]['Topic']
    content = df.iloc[best_id]['Content']
    code = df.iloc[best_id]['Code']

    return theme, content, code


In [ ]:
import ollama
def ask_llm(question):
    theme, content, code = ask_question(question)

    instruction = f"""
    You are a professional Django Mentor. 
    Use ONLY the provided documentation context to answer the user's question.
    
    GUIDELINES:
    1. Answer in POLISH (po polsku).
    2. If the answer is in the documentation, explain it using the context and the code example.
    3. If the documentation is not related to the question at all, say: "Nie znalazłem tego w lokalnej bazie."

    DOCUMENTATION CONTEXT:
    {content}
    
    CODE EXAMPLE:
    {code}
    
    USER QUESTION: 
    {question}"""


    print(content)
     
    
    response = ollama.chat(model='llama3.2:3b', messages=[
        {
            'role': 'user',
            'content': instruction
        },
    ])

    return response['message']['content']

print(ask_llm('how we can Use the model'))


In [ ]:
def test_search(query):
    t, cont, c = ask_question(query, certainty_trashold=0.01)
    print(f"--- TEST DLA: {query} ---")
    print(f"ZNALEZIONY TEMAT: {t}")
    print(f"DŁUGOŚĆ TREŚCI: {len(cont)} znaków")
    
    # 2. Wyślij to do LLM
    answer = ask_llm(query)
    print(f"\nODPOWIEDŹ LLAMA:\n{answer}")

test_search('field types')
